AI Software Engineer Agent
Combining Code Review & Automated Test Generation **bold text**

In [47]:
# Standard Library
import os
import subprocess
import shutil
from pathlib import Path
from google.colab import userdata
from tqdm.notebook import tqdm
import shutil
# Git
from git import Repo

# OpenAI
from openai import OpenAI

In [48]:
from google.colab import userdata
from openai import OpenAI

# Get the API key from Colab Secrets
OPENAI_API_KEY = userdata.get("Code_Review_Test_Key")

# Initialize the OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Define the model used by all AI agents
MODEL = "gpt-5.5"

print("✅ OpenAI client initialized successfully.")

✅ OpenAI client initialized successfully.


# =====================================================
# 🤖 AGENT 1 - REPOSITORY MANAGER
# =====================================================

## Responsibilities
- Clone the GitHub repository
- Prepare the local workspace
- Locate all Python source files

Clone a GitHub repository into a local directory. If the destination already exists, it is removed before cloning a fresh copy.

In [49]:
from git import Repo
from pathlib import Path
import shutil

# Clone a GitHub repository into a local directory
def clone_repository(repository_url: str, destination: str):

    # Create a Path object for the destination folder
    destination_path = Path(destination)

    # Remove the existing repository if it already exists
    if destination_path.exists():
        shutil.rmtree(destination_path)

    # Clone the repository from GitHub
    Repo.clone_from(repository_url, destination)

In [50]:
# GitHub repository URL
repository_url = "https://github.com/secksaloumdieng-pixel/benchmark_inventory_app.git"

# Local destination folder
repository_path = "repositories/project"

# Clone the repository
clone_repository(repository_url, repository_path)

print("✅ Repository cloned successfully.")

✅ Repository cloned successfully.


This function scans the repository recursively and returns all Python (.py) files. These files will later be analyzed by the AI Code Reviewer and Test Generator.

In [51]:
# Find all Python files in the repository
def find_python_files(repository_path: str):

    # Search recursively for all .py files
    python_files = list(Path(repository_path).rglob("*.py"))

    return python_files

# Detect the Python import root
def find_import_root(repository_path: str) -> Path:

    repository = Path(repository_path)

    if (repository / "src").exists():
        return repository / "src"

    return repository


In [52]:
# Find all Python files
python_files = find_python_files(repository_path)

# Display the files found
print("Python files found:\n")

for file in python_files:
    print(file)

Python files found:

repositories/project/app.py
repositories/project/__init__.py
repositories/project/utils/dates.py
repositories/project/utils/validation.py
repositories/project/utils/__init__.py
repositories/project/reports/__init__.py
repositories/project/reports/inventory_report.py
repositories/project/storage/json_store.py
repositories/project/storage/__init__.py
repositories/project/services/inventory_service.py
repositories/project/services/__init__.py
repositories/project/models/transaction.py
repositories/project/models/__init__.py
repositories/project/models/inventory_item.py
repositories/project/models/product.py


This function reads the content of a Python source file and returns it as a string. The returned code will be sent to the AI Code Reviewer and the AI Test Generator.

In [53]:
# Read the content of a Python source file
def read_python_file(file_path: Path):

    # Open the file in read mode
    with open(file_path, "r") as file:

        # Return the file content as a string
        return file.read()

In [54]:
# Store the source code of each Python file
source_files = []

# Read every Python file found
for python_file in python_files:

    source_code = read_python_file(python_file)

    source_files.append({
        "path": python_file,
        "source_code": source_code,
    })

print(f"✅ {len(source_files)} Python files loaded successfully.")

✅ 15 Python files loaded successfully.


# =====================================================
# 🔍 AGENT 2 – AI Code Reviewer
# =====================================================

## Responsibilities
- Analyze Python source code
- Detect potential issues
- Suggest improvements
- Generate a review report

Initialize the OpenAI client to enable communication with the language model. This client will be used throughout the notebook for code review, test generation, and feedback.

This function sends Python source code to the LLM and requests a professional code review. The model analyzes the code and returns suggestions, potential issues, and best practice recommendations.

In [55]:
# System prompt for the AI Code Reviewer
CODE_REVIEW_PROMPT = """

You are a Senior Python Software Engineer.

Review the provided Python code and generate a concise report.

Use the following format:

==============================
AI CODE REVIEW
==============================

Summary:
- Briefly describe what the code does.

Issues:
- List any bugs, code smells, security concerns, or performance issues.
- If no issues are found, write: None.

Suggestions:
- Provide practical recommendations to improve the code.
- If no improvements are needed, write: None.

Overall:
- Give a one-sentence assessment of the code quality.

Keep the review clear, concise, and under 150 words.
"""

# Review Python source code using the OpenAI model
def review_code(source_code: str):

    # Send the source code to the LLM
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": CODE_REVIEW_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Please review the following Python code.

```python
{source_code}
```
""",
            },
        ],
    )

    # Return the AI-generated review
    return response.choices[0].message.content

In [56]:
# Create the reviews directory
reviews_dir = Path("workspace/reviews")
reviews_dir.mkdir(parents=True, exist_ok=True)

report_path = reviews_dir / "code_review_report.md"

code_reviews = []

with open(report_path, "w", encoding="utf-8") as report:

    report.write("# AI Code Review Report\n\n")

    for file_data in tqdm(source_files, desc="Reviewing Python files"):

        file_path = file_data["path"]
        source_code = file_data["source_code"]

        review = review_code(source_code)

        code_reviews.append({
            "path": file_path,
            "review": review,
        })

        report.write(f"## {file_path}\n\n")
        report.write(review)
        report.write("\n\n---\n\n")

print(f"✅ Code review report saved to: {report_path}")

Reviewing Python files:   0%|          | 0/15 [00:00<?, ?it/s]

✅ Code review report saved to: workspace/reviews/code_review_report.md


=====================================================

# 🧪 AGENT 3 - AI TEST GENERATOR

=====================================================

Responsibilities

- Generate pytest unit tests

- Cover normal cases and edge cases

- Use the code review feedback

- Return valid Python test code

In [57]:
# Instructions for the AI test generation agent
TEST_GENERATION_PROMPT = """
You are a Senior Python Test Engineer.

Your task is to generate high-quality pytest unit tests.

Requirements:
- Use pytest.
- Cover normal cases.
- Cover edge cases.
- Use the code review findings to improve the tests.
- Return only valid Python code.
- Do not include explanations.
- Do not use Markdown code fences.
"""

# Generate pytest tests using the source code, review, and module name
def generate_tests(source_code: str, review: str, module_name: str):

    # Send the source code and module information to the LLM
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": TEST_GENERATION_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Source Code:

{source_code}

Code Review:

{review}

Python module to import from:

{module_name}

Generate pytest unit tests.
Import functions only from `{module_name}`.
Do not import from `main`.
""",
            },
        ],
    )

    # Return the generated test code
    return response.choices[0].message.content

In [58]:
# Create a clean generated tests directory
tests_dir = Path("workspace/generated_tests")

if tests_dir.exists():
    shutil.rmtree(tests_dir)

tests_dir.mkdir(parents=True, exist_ok=True)

generated_tests = []
import_root = find_import_root(repository_path)

# Generate tests for each relevant source file
for file_data, review_data in tqdm(
    zip(source_files, code_reviews),
    total=len(source_files),
    desc="Generating pytest tests"
):

    file_path = file_data["path"]
    source_code = file_data["source_code"]
    review = review_data["review"]

    # Skip files that should not receive generated tests
    if file_path.name == "__init__.py":
        continue

    if file_path.name.startswith("test_"):
        continue

    if file_path.name == "noxfile.py":
        continue

    if not source_code.strip():
        continue

    # Convert src/sample/simple.py into sample.simple
    relative_module_path = file_path.relative_to(import_root)
    module_name = ".".join(relative_module_path.with_suffix("").parts)

    # Generate tests with the correct import path
    test_code = generate_tests(source_code, review, module_name)

    # Build the generated test file name
    relative_path = file_path.relative_to(repository_path)
    test_name = "test_" + "_".join(relative_path.with_suffix("").parts) + ".py"
    test_path = tests_dir / test_name

    # Save the generated tests
    with open(test_path, "w", encoding="utf-8") as test_file:
        test_file.write(test_code)

    generated_tests.append({
    "source_path": str(file_path),
    "test_path": str(test_path),
    "module_name": module_name,
    "test_code": test_code,
    })

print(f"✅ {len(generated_tests)} test files generated successfully.")
print(f"📁 Saved in: {tests_dir}")

Generating pytest tests:   0%|          | 0/15 [00:00<?, ?it/s]

✅ 9 test files generated successfully.
📁 Saved in: workspace/generated_tests


=====================================================

# ⚙️ AGENT 4 - TEST RUNNER

=====================================================

## Responsibilities

- Execute the generated pytest files
- Capture test results and errors
- Report passed and failed tests
- Prepare feedback for test improvement

In [59]:

# Run all generated tests with pytest
def run_tests():

    # Configure the Python import path
    env = os.environ.copy()
    import_root = find_import_root(repository_path)
    env["PYTHONPATH"] = str(import_root)

    # Execute pytest and capture the output
    result = subprocess.run(
        [
            "pytest",
            str(tests_dir),
            "-v",
        ],
        capture_output=True,
        text=True,
        env=env,
    )

    # Combine standard output and errors
    output = result.stdout + result.stderr

    # Return the test status and full output
    return result.returncode == 0, output

In [60]:
# Create the reports directory
reports_dir = Path("workspace/reports")
reports_dir.mkdir(parents=True, exist_ok=True)

summary_report_path = reports_dir / "test_results.md"
full_report_path = reports_dir / "test_results_full.txt"

# Run the generated tests
tests_passed, test_output = run_tests()

# Save the complete pytest output
with open(full_report_path, "w", encoding="utf-8") as report:
    report.write(test_output)

# Extract test statistics
def get_stat(name: str) -> int:
    match = re.search(rf"(\d+)\s+{name}", test_output)
    return int(match.group(1)) if match else 0


passed_count = get_stat("passed")
failed_count = get_stat("failed")
error_count = get_stat("errors?")
skipped_count = get_stat("skipped")
xfailed_count = get_stat("xfailed")

# Extract failed and errored test names
problem_tests = re.findall(
    r"^(FAILED|ERROR)\s+([^\s]+)",
    test_output,
    re.MULTILINE,
)

# Create a readable Markdown summary
with open(summary_report_path, "w", encoding="utf-8") as report:
    report.write("# Pytest Results\n\n")

    report.write("## Summary\n\n")
    report.write(f"- Passed: {passed_count}\n")
    report.write(f"- Failed: {failed_count}\n")
    report.write(f"- Errors: {error_count}\n")
    report.write(f"- Skipped: {skipped_count}\n")
    report.write(f"- Expected failures: {xfailed_count}\n\n")

    report.write("## Status\n\n")

    if tests_passed:
        report.write("✅ All tests passed successfully.\n\n")
    else:
        report.write("❌ Some tests failed or produced errors.\n\n")

    report.write("## Failed Tests\n\n")

    if not problem_tests:
        report.write("None.\n")
    else:
        for status, test_name in problem_tests:
            report.write(f"- **{status}** — `{test_name}`\n")

    report.write("\n## Full Output\n\n")
    report.write(f"See `{full_report_path}` for the complete pytest output.\n")

print(f"✅ Summary report saved to: {summary_report_path}")
print(f"📄 Full pytest output saved to: {full_report_path}")

✅ Summary report saved to: workspace/reports/test_results.md
📄 Full pytest output saved to: workspace/reports/test_results_full.txt


=====================================================

# 🔧 AGENT 5 - CODE FIXER

=====================================================

## Responsibilities

- Analyze failed pytest results
- Identify whether the problem comes from the source code
- Generate corrected Python code
- Save corrected files in a separate folder
- Keep the original repository unchanged

In [61]:
# Identify the source files associated with failed tests
failed_sources = []

for test in generated_tests:

    if test["test_path"] in test_output:

        failed_sources.append({
            "source_path": test["source_path"],
            "module_name": test["module_name"],
        })

print("Failed source files:")

if not failed_sources:
    print("✅ No source files require fixes.")
else:
    for file in failed_sources:
        print(f"- {file['source_path']}")

Failed source files:
- repositories/project/app.py
- repositories/project/utils/dates.py
- repositories/project/utils/validation.py
- repositories/project/reports/inventory_report.py
- repositories/project/storage/json_store.py
- repositories/project/services/inventory_service.py
- repositories/project/models/transaction.py
- repositories/project/models/inventory_item.py
- repositories/project/models/product.py


In [62]:
CODE_FIX_PROMPT = """
You are a Senior Python Software Engineer.

Your task is to fix Python source code using:

- The original source code
- The code review
- The pytest failure output

Requirements:
- Fix only the source code.
- Do not modify the tests.
- Preserve the original behavior when possible.
- Return only valid Python code.
- Do not include explanations.
- Do not use Markdown code fences.
"""

# Generate corrected Python source code
def fix_code(
    source_code: str,
    review: str,
    test_output: str,
    module_name: str,
):

    # Send the source code and failure details to the LLM
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": CODE_FIX_PROMPT,
            },
            {
                "role": "user",
                "content": f"""
Python module:

{module_name}

Original source code:

{source_code}

Code review:

{review}

Pytest output:

{test_output}

Return the corrected source code.
""",
            },
        ],
    )

    # Return the corrected source code
    return response.choices[0].message.content

In [63]:
# Create the fixed code directory
fixed_code_dir = Path("workspace/fixed_code")

if fixed_code_dir.exists():
    shutil.rmtree(fixed_code_dir)

fixed_code_dir.mkdir(parents=True, exist_ok=True)

fixed_files = []

# Stop if all tests passed
if tests_passed:
    print("✅ All tests passed. No code fixes are required.")

else:
    # Fix only source files linked to failed tests
    for failed in tqdm(
        failed_sources,
        desc="Fixing failed source files"
    ):

        source_path = failed["source_path"]
        module_name = failed["module_name"]

        # Find the original source code
        file_data = next(
            item for item in source_files
            if str(item["path"]) == source_path
        )

        # Find the corresponding code review
        review_data = next(
            item for item in code_reviews
            if str(item["path"]) == source_path
        )

        # Generate the corrected source code
        fixed_code = fix_code(
            source_code=file_data["source_code"],
            review=review_data["review"],
            test_output=test_output,
            module_name=module_name,
        )

        # Preserve the original folder structure
        relative_path = Path(source_path).relative_to(repository_path)
        output_path = fixed_code_dir / relative_path
        output_path.parent.mkdir(parents=True, exist_ok=True)

        # Save the corrected file
        with open(output_path, "w", encoding="utf-8") as file:
            file.write(fixed_code)

        fixed_files.append(output_path)

    print(f"✅ {len(fixed_files)} corrected file(s) saved.")
    print(f"📁 Saved in: {fixed_code_dir}")

Fixing failed source files:   0%|          | 0/9 [00:00<?, ?it/s]

✅ 9 corrected file(s) saved.
📁 Saved in: workspace/fixed_code


In [64]:
# Create the fix report directory
reports_dir = Path("workspace/reports")
reports_dir.mkdir(parents=True, exist_ok=True)

fix_report_path = reports_dir / "code_fix_report.md"

with open(fix_report_path, "w", encoding="utf-8") as report:

    report.write("# AI Code Fix Report\n\n")

    if not fixed_files:

        report.write("## Summary\n\n")
        report.write("No code fixes were required.\n")

    else:

        report.write("## Corrected Files\n\n")

        for file in fixed_files:
            report.write(f"- {file}\n")

print("=" * 60)
print("CODE FIX SUMMARY")
print("=" * 60)

if not fixed_files:
    print("✅ No code fixes were required.")
else:
    print(f"✅ {len(fixed_files)} corrected file(s) generated.")

print(f"📄 Report saved to: {fix_report_path}")
print(f"📁 Corrected files: {fixed_code_dir}")

CODE FIX SUMMARY
✅ 9 corrected file(s) generated.
📄 Report saved to: workspace/reports/code_fix_report.md
📁 Corrected files: workspace/fixed_code


=====================================================

# ✅ AGENT 6 - VALIDATION AGENT

=====================================================

Responsibilities

• Execute the corrected source code
• Re-run the generated pytest tests
• Compare test results before and after the fixes
• Generate the final validation report

In [65]:
# Run pytest against the corrected source code
def validate_fixed_code():

    env = os.environ.copy()

    import_root = fixed_code_dir / "src"

    if not import_root.exists():
        import_root = fixed_code_dir

    env["PYTHONPATH"] = str(import_root)

    result = subprocess.run(
        [
            "pytest",
            str(tests_dir),
            "-v",
        ],
        capture_output=True,
        text=True,
        env=env,
    )

    output = result.stdout + result.stderr

    return result.returncode == 0, output

In [66]:
# Validate the corrected source code

if not fixed_files:

    validation_passed = tests_passed
    validation_output = test_output

    print("✅ No corrected files to validate.")
    print("Using the original test results.")

else:

    validation_passed, validation_output = validate_fixed_code()

    print(validation_output)

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: anyio-4.14.2, langsmith-0.10.2, typeguard-4.5.2
collecting ... collected 261 items

workspace/generated_tests/test_app.py::test_build_parser_parses_add_command_with_all_options PASSED [  0%]
workspace/generated_tests/test_app.py::test_build_parser_uses_add_defaults PASSED [  0%]
workspace/generated_tests/test_app.py::test_build_parser_requires_command PASSED [  1%]
workspace/generated_tests/test_app.py::test_build_parser_rejects_missing_required_arguments[argv0] PASSED [  1%]
workspace/generated_tests/test_app.py::test_build_parser_rejects_missing_required_arguments[argv1] PASSED [  1%]
workspace/generated_tests/test_app.py::test_build_parser_rejects_missing_required_arguments[argv2] PASSED [  2%]
workspace/generated_tests/test_app.py::test_main_add_creates_product_and_passes_

In [67]:
from pathlib import Path

validation_report = reports_dir / "validation_report.md"

with open(validation_report, "w", encoding="utf-8") as report:

    report.write("# Validation Report\n\n")

    report.write("## Original Test Result\n\n")
    report.write(test_output)
    report.write("\n\n")

    report.write("## Validation Result\n\n")
    report.write(validation_output)

print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)

if validation_passed:
    print("✅ All corrected tests passed.")
else:
    print("❌ Some tests are still failing.")

print(f"📄 Validation report saved to: {validation_report}")

VALIDATION SUMMARY
✅ All corrected tests passed.
📄 Validation report saved to: workspace/reports/validation_report.md


In [68]:
import re
from pathlib import Path

# Count project statistics
files_reviewed = len(source_files)
tests_generated = len(generated_tests)

# Extract pytest statistics
def extract_test_stats(output):

    passed = 0
    failed = 0
    errors = 0
    skipped = 0
    xfailed = 0

    match = re.search(
        r"=+\s*(.*?)\s*in\s*[\d\.]+s\s*=+",
        output,
        re.DOTALL,
    )

    if match:

        summary = match.group(1)

        passed_match = re.search(r"(\d+)\s+passed", summary)
        failed_match = re.search(r"(\d+)\s+failed", summary)
        errors_match = re.search(r"(\d+)\s+errors?", summary)
        skipped_match = re.search(r"(\d+)\s+skipped", summary)
        xfailed_match = re.search(r"(\d+)\s+xfailed", summary)

        if passed_match:
            passed = int(passed_match.group(1))

        if failed_match:
            failed = int(failed_match.group(1))

        if errors_match:
            errors = int(errors_match.group(1))

        if skipped_match:
            skipped = int(skipped_match.group(1))

        if xfailed_match:
            xfailed = int(xfailed_match.group(1))

    return passed, failed, errors, skipped, xfailed


before_passed, before_failed, before_errors, before_skipped, before_xfailed = extract_test_stats(test_output)

after_passed, after_failed, after_errors, after_skipped, after_xfailed = extract_test_stats(validation_output)

repository_name = Path(repository_path).name

final_report = reports_dir / "final_ai_report.md"

with open(final_report, "w", encoding="utf-8") as report:

    report.write("# Final AI Software Engineer Report\n\n")

    report.write(f"Repository: {repository_name}\n\n")

    report.write("## Project Summary\n\n")
    report.write(f"- Files Reviewed: {files_reviewed}\n")
    report.write(f"- Tests Generated: {tests_generated}\n\n")

    report.write("## Before Fix\n\n")
    report.write(f"- Passed: {before_passed}\n")
    report.write(f"- Failed: {before_failed}\n")
    report.write(f"- Errors: {before_errors}\n")
    report.write(f"- Skipped: {before_skipped}\n\n")

    report.write("## After Fix\n\n")
    report.write(f"- Passed: {after_passed}\n")
    report.write(f"- Failed: {after_failed}\n")
    report.write(f"- Errors: {after_errors}\n")
    report.write(f"- Skipped: {after_skipped}\n\n")

    report.write("## Repository Status\n\n")

    if after_failed == 0 and after_errors == 0:
        report.write("✅ Repository successfully validated.\n")
    else:
        report.write("❌ Repository still contains failing tests.\n")

print(f"✅ Final report saved to: {final_report}")

✅ Final report saved to: workspace/reports/final_ai_report.md
